# LangfuseEvaluationDataset — E2E Tests (Local Mode)

Manual E2E scenarios for `LangfuseEvaluationDataset` with `sync_policy="local"`.

**Prerequisites:**
- Langfuse credentials available as env vars: `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`
- `langfuse>=3.14.0` installed

**After each scenario**, verify the results in the **Langfuse UI** — look for the
dataset named `e2e-local-<timestamp>` that gets printed in the setup cell.

| Scenario | What it tests |
|----------|---------------|
| 1 | Fresh start — local file seeds a new remote dataset |
| 2 | Idempotent reload — repeated `load()` produces no duplicates |
| 3 | Incremental sync — new item in local file is uploaded, old items untouched |
| 4 | `save()` — uploads new items to remote, merges into local, skips duplicates |

---
## Setup

In [1]:
import json
import logging
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import yaml

# Ensure the src directory is importable when running from notebooks/
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = str(PROJECT_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from kedro_agentic_workflows.datasets.langfuse_evaluation_dataset import (
    LangfuseEvaluationDataset,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s — %(levelname)s — %(message)s",
)

# Load credentials from the Kedro local config
CREDENTIALS_FILE = Path("../conf/local/credentials.yml")
with open(CREDENTIALS_FILE) as f:
    all_credentials = yaml.safe_load(f)

CREDENTIALS = {
    "public_key": all_credentials["langfuse_credentials"]["public_key"],
    "secret_key": all_credentials["langfuse_credentials"]["secret_key"],
    "host": all_credentials["langfuse_credentials"]["host"],
}

# Unique name per run — avoids cleanup issues
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
DATASET_NAME = f"e2e-local-{RUN_ID}"

TMP_DIR = Path(tempfile.mkdtemp(prefix="e2e_langfuse_"))
LOCAL_FILE = TMP_DIR / "eval_items.json"

print(f"Dataset name : {DATASET_NAME}")
print(f"Local file   : {LOCAL_FILE}")

/Users/elena_khaustova/opt/miniconda3/envs/langfuse-eval-env/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Dataset name : e2e-local-20260313-115313
Local file   : /var/folders/c9/fg0s1kfd1sdgjz4blby9mt3r0000gn/T/e2e_langfuse_9nz9h6s1/eval_items.json


## Test data

Synthetic evaluation items reused across all four scenarios.

In [2]:
INITIAL_ITEMS = [
    {
        "id": "e2e_001",
        "input": {"question": "How do I reset my password?"},
        "expected_output": {"intent": "account_management"},
    },
    {
        "id": "e2e_002",
        "input": {"question": "My order hasn't arrived yet."},
        "expected_output": {"intent": "order_status"},
    },
    {
        "id": "e2e_003",
        "input": {"question": "Can I get a refund?"},
        "expected_output": {"intent": "refund_request"},
    },
]

# Single new item added in Scenario 3
NEW_ITEM = {
    "id": "e2e_004",
    "input": {"question": "I want to cancel my subscription."},
    "expected_output": {"intent": "cancellation"},
}

# Items passed to save() in Scenario 4:
#   e2e_005 — new
#   e2e_006 — new
#   e2e_001 — DUPLICATE with different content (must be skipped everywhere)
SAVE_ITEMS = [
    {
        "id": "e2e_005",
        "input": {"question": "Where is your nearest store?"},
        "expected_output": {"intent": "store_locator"},
    },
    {
        "id": "e2e_006",
        "input": {"question": "Do you ship internationally?"},
        "expected_output": {"intent": "shipping_info"},
    },
    {
        "id": "e2e_001",
        "input": {"question": "SHOULD NOT OVERWRITE THE ORIGINAL"},
        "expected_output": {"intent": "SHOULD_NOT_APPEAR"},
    },
]

print("Test data defined ✓")

Test data defined ✓


## Helpers

In [3]:
def write_local(items: list[dict]) -> None:
    """Write items to the local JSON test file."""
    LOCAL_FILE.write_text(json.dumps(items, indent=2))
    print(f"  Wrote {len(items)} item(s) to local file")


def read_local() -> list[dict]:
    """Read items from the local JSON test file."""
    return json.loads(LOCAL_FILE.read_text())


def make_dataset() -> LangfuseEvaluationDataset:
    """Create a dataset instance with the shared test config."""
    return LangfuseEvaluationDataset(
        dataset_name=DATASET_NAME,
        credentials=CREDENTIALS,
        filepath=str(LOCAL_FILE),
        sync_policy="local",
    )


def summarise_remote(dataset_client) -> dict:
    """Return a summary dict of the remote dataset for inspection."""
    return {
        "count": len(dataset_client.items),
        "ids": sorted(item.id for item in dataset_client.items),
    }

---
## Scenario 1: Fresh start — create remote dataset + sync local items

**Steps:**
1. Write 3 items (`e2e_001`, `e2e_002`, `e2e_003`) to the local JSON file.
2. Instantiate `LangfuseEvaluationDataset` with `sync_policy="local"`.
3. Call `load()`.

**Expected behaviour:**
- Remote dataset is created (it didn't exist before).
- All 3 local items are uploaded to remote.
- `load()` returns a `DatasetClient` with 3 items.

**Expected logs:**
```
Dataset 'e2e-local-...' not found on Langfuse, creating it.
Syncing 3 new item(s) from '...' to remote dataset 'e2e-local-...'.
Loaded dataset 'e2e-local-...' with 3 item(s) (sync_policy='local').
```

**Verify in Langfuse UI:**
- Dataset `e2e-local-<timestamp>` exists under Datasets.
- Contains exactly 3 items: `e2e_001`, `e2e_002`, `e2e_003`.
- Each item has the correct `input` and `expected_output`.

In [4]:
print("=" * 60)
print("SCENARIO 1: Fresh start")
print("=" * 60)

# Step 1: write local file
write_local(INITIAL_ITEMS)

# Step 2-3: create dataset + load
ds = make_dataset()
result = ds.load()

# Assertions
summary = summarise_remote(result)
assert summary["count"] == 3, f"Expected 3 items, got {summary['count']}"
assert summary["ids"] == ["e2e_001", "e2e_002", "e2e_003"], (
    f"Unexpected ids: {summary['ids']}"
)

print(f"\n✓ PASSED — Remote has {summary['count']} items: {summary['ids']}")
print("  → Check Langfuse UI: dataset should exist with 3 items")

SCENARIO 1: Fresh start
  Wrote 3 item(s) to local file


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Dataset 'e2e-local-20260313-115313' not found on Langfuse, creating it.
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Syncing 3 new item(s) from '/var/folders/c9/fg0s1kfd1sdgjz4blby9mt3r0000gn/T/e2e_langfuse_9nz9h6s1/eval_items.json' to remote dataset 'e2e-local-20260313-115313'.
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 3 item(s) (sync_policy='local').



✓ PASSED — Remote has 3 items: ['e2e_001', 'e2e_002', 'e2e_003']
  → Check Langfuse UI: dataset should exist with 3 items


---
## Scenario 2: Idempotent reload — no duplicates on repeated `load()`

**Steps:**
1. Create a new `LangfuseEvaluationDataset` instance (same config as Scenario 1).
2. Call `load()` again without changing the local file.

**Expected behaviour:**
- No new items are uploaded (all 3 already exist on remote).
- `load()` returns a `DatasetClient` with exactly 3 items.

**Expected logs:**
```
Loaded dataset 'e2e-local-...' with 3 item(s) (sync_policy='local').
```
No "Syncing" log line should appear.

**Verify in Langfuse UI:**
- Still exactly 3 items (not 6).

In [5]:
print("=" * 60)
print("SCENARIO 2: Idempotent reload")
print("=" * 60)

# Same local file, fresh dataset instance
ds2 = make_dataset()
result2 = ds2.load()

# Assertions
summary2 = summarise_remote(result2)
assert summary2["count"] == 3, f"Expected 3 items, got {summary2['count']}"
assert summary2["ids"] == ["e2e_001", "e2e_002", "e2e_003"], (
    f"Unexpected ids: {summary2['ids']}"
)

print(f"\n✓ PASSED — Still {summary2['count']} items (no duplicates)")
print("  → Check Langfuse UI: still exactly 3 items, no duplicates")

SCENARIO 2: Idempotent reload


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 3 item(s) (sync_policy='local').



✓ PASSED — Still 3 items (no duplicates)
  → Check Langfuse UI: still exactly 3 items, no duplicates


---
## Scenario 3: Incremental sync — one new local item

**Steps:**
1. Append a new item (`e2e_004`) to the local file (so it now has 4 items).
2. Create a new `LangfuseEvaluationDataset` instance and call `load()`.

**Expected behaviour:**
- Only 1 new item (`e2e_004`) is uploaded to remote.
- The 3 existing items remain unchanged.
- `load()` returns a `DatasetClient` with 4 items.

**Expected logs:**
```
Syncing 1 new item(s) from '...' to remote dataset 'e2e-local-...'.
Loaded dataset 'e2e-local-...' with 4 item(s) (sync_policy='local').
```

**Verify in Langfuse UI:**
- 4 items total: `e2e_001` – `e2e_004`.
- `e2e_001` – `e2e_003` have their original content unchanged.

In [6]:
print("=" * 60)
print("SCENARIO 3: Incremental sync")
print("=" * 60)

# Step 1: add one new item to local file
write_local(INITIAL_ITEMS + [NEW_ITEM])

# Step 2: load
ds3 = make_dataset()
result3 = ds3.load()

# Assertions
summary3 = summarise_remote(result3)
assert summary3["count"] == 4, f"Expected 4 items, got {summary3['count']}"
assert "e2e_004" in summary3["ids"], "New item e2e_004 not found on remote"

# Verify original item content is unchanged
item_001 = next(i for i in result3.items if i.id == "e2e_001")
assert item_001.input["question"] == "How do I reset my password?", (
    f"Original item was modified: {item_001.input}"
)

print(f"\n✓ PASSED — {summary3['count']} items: {summary3['ids']}")
print(f"  e2e_001 content preserved: '{item_001.input['question']}'")
print("  → Check Langfuse UI: 4 items, original content intact")

SCENARIO 3: Incremental sync
  Wrote 4 item(s) to local file


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Syncing 1 new item(s) from '/var/folders/c9/fg0s1kfd1sdgjz4blby9mt3r0000gn/T/e2e_langfuse_9nz9h6s1/eval_items.json' to remote dataset 'e2e-local-20260313-115313'.
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 4 item(s) (sync_policy='local').



✓ PASSED — 4 items: ['e2e_001', 'e2e_002', 'e2e_003', 'e2e_004']
  e2e_001 content preserved: 'How do I reset my password?'
  → Check Langfuse UI: 4 items, original content intact


---
## Scenario 4: `save()` — upload new items, merge to local, skip duplicates

**Steps:**
1. Call `save(SAVE_ITEMS)` where `SAVE_ITEMS` contains:
   - `e2e_005` — new item
   - `e2e_006` — new item
   - `e2e_001` — duplicate of existing item **with different content** (should be skipped)
2. Call `load()` to get the final remote state.
3. Read the local file to verify the merge.

**Expected behaviour — remote:**
- Only `e2e_005` and `e2e_006` are uploaded (2 new items).
- `e2e_001` is **not** overwritten — original content preserved.
- Total: 6 items on remote.

**Expected behaviour — local file:**
- Merged content: 4 existing + 2 new = 6 items.
- Duplicate `e2e_001` from `SAVE_ITEMS` is dropped (existing takes precedence).
- `e2e_001` retains its original content ("How do I reset my password?").

**Expected logs:**
```
Uploading 2 new item(s) to remote dataset 'e2e-local-...'.
```

**Verify in Langfuse UI:**
- 6 items: `e2e_001` – `e2e_006`.
- `e2e_001` still shows "How do I reset my password?" (not "SHOULD NOT OVERWRITE...").

In [7]:
print("=" * 60)
print("SCENARIO 4: save() with new items + duplicate")
print("=" * 60)

# Step 1: save 2 new + 1 duplicate
ds4 = make_dataset()
ds4.save(SAVE_ITEMS)

# Step 2: reload remote to verify
result4 = ds4.load()
summary4 = summarise_remote(result4)

expected_ids = ["e2e_001", "e2e_002", "e2e_003", "e2e_004", "e2e_005", "e2e_006"]
assert summary4["count"] == 6, f"Expected 6 remote items, got {summary4['count']}"
assert summary4["ids"] == expected_ids, f"Unexpected remote ids: {summary4['ids']}"

# Verify e2e_001 was NOT overwritten on remote
item_001_remote = next(i for i in result4.items if i.id == "e2e_001")
assert item_001_remote.input["question"] == "How do I reset my password?", (
    f"Remote e2e_001 was overwritten! Got: {item_001_remote.input}"
)
print(f"  Remote: {summary4['count']} items, e2e_001 content preserved ✓")

# Step 3: verify local file
local_items = read_local()
local_ids = sorted(item["id"] for item in local_items)
assert len(local_items) == 6, f"Expected 6 local items, got {len(local_items)}"
assert local_ids == expected_ids, f"Unexpected local ids: {local_ids}"

# Verify e2e_001 was NOT overwritten in local file
item_001_local = next(i for i in local_items if i["id"] == "e2e_001")
assert item_001_local["input"]["question"] == "How do I reset my password?", (
    f"Local e2e_001 was overwritten! Got: {item_001_local['input']}"
)
print(f"  Local:  {len(local_items)} items, e2e_001 content preserved ✓")

print(f"\n✓ PASSED — Remote and local both have 6 items, duplicate skipped")
print("  → Check Langfuse UI: 6 items, e2e_001 shows original content")

kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Uploading 2 new item(s) to remote dataset 'e2e-local-20260313-115313'.


SCENARIO 4: save() with new items + duplicate


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 6 item(s) (sync_policy='local').


  Remote: 6 items, e2e_001 content preserved ✓
  Local:  6 items, e2e_001 content preserved ✓

✓ PASSED — Remote and local both have 6 items, duplicate skipped
  → Check Langfuse UI: 6 items, e2e_001 shows original content


---
## Local mode summary

If all four cells above printed `✓ PASSED`, the local mode is working correctly.

**Langfuse UI checklist for dataset `e2e-local-<timestamp>`:**

| Check | Expected |
|-------|----------|
| Dataset exists | Yes |
| Total items | 6 |
| Item ids | `e2e_001` – `e2e_006` |
| `e2e_001` input | `{"question": "How do I reset my password?"}` |
| No duplicates | Each id appears exactly once |

---
# Remote Mode (Scenarios 5–7)

These scenarios test `sync_policy="remote"` where the Langfuse dataset is the
sole source of truth. They **reuse the dataset created by Scenarios 1–4** above
(which should have 6 items on Langfuse at this point).

| Scenario | What it tests |
|----------|---------------|
| 5 | Load from existing remote — no local file interaction |
| 6 | `save()` in remote mode — uploads to remote, no local file written |
| 7 | Versioned load — pin to a historical snapshot |

---
## Scenario 5: Remote mode — load from existing dataset

**Steps:**
1. Instantiate `LangfuseEvaluationDataset` with `sync_policy="remote"` and **no `filepath`**.
2. Call `load()`.

**Expected behaviour:**
- No dataset creation (it already exists from Scenarios 1–4).
- No local file interaction at all.
- `load()` returns a `DatasetClient` with all 6 items from the remote dataset.

**Expected logs:**
```
Loaded dataset 'e2e-local-...' with 6 item(s) (sync_policy='remote').
```
No "Syncing" or "creating" log lines.

**Verify in Langfuse UI:**
- Dataset unchanged — still 6 items.

In [8]:
print("=" * 60)
print("SCENARIO 5: Remote mode — load from existing dataset")
print("=" * 60)

ds5 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)

result5 = ds5.load()
summary5 = summarise_remote(result5)

assert summary5["count"] == 6, f"Expected 6 items, got {summary5['count']}"
expected_ids = ["e2e_001", "e2e_002", "e2e_003", "e2e_004", "e2e_005", "e2e_006"]
assert summary5["ids"] == expected_ids, f"Unexpected ids: {summary5['ids']}"

# Spot-check content
item_003 = next(i for i in result5.items if i.id == "e2e_003")
assert item_003.input["question"] == "Can I get a refund?", (
    f"Unexpected content for e2e_003: {item_003.input}"
)

print(f"\n✓ PASSED — Remote mode loaded {summary5['count']} items: {summary5['ids']}")
print("  No local file was read or written")
print("  → Check Langfuse UI: dataset unchanged, still 6 items")

kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 6 item(s) (sync_policy='remote').


SCENARIO 5: Remote mode — load from existing dataset

✓ PASSED — Remote mode loaded 6 items: ['e2e_001', 'e2e_002', 'e2e_003', 'e2e_004', 'e2e_005', 'e2e_006']
  No local file was read or written
  → Check Langfuse UI: dataset unchanged, still 6 items


---
## Scenario 6: Remote mode — `save()` uploads to remote, no local file

**Steps:**
1. Instantiate `LangfuseEvaluationDataset` with `sync_policy="remote"` (no `filepath`).
2. Call `save()` with a new item.
3. Call `load()` to verify the item was uploaded to remote.
4. Verify the local file was NOT modified (still has 6 items from Scenario 4).

**Expected behaviour:**
- `save()` uploads the new item to remote.
- No local file is read or written.
- `load()` returns 7 items (6 existing + 1 new).

**Expected logs:**
```
Uploading 1 new item(s) to remote dataset 'e2e-local-...'.
```

**Verify in Langfuse UI:**
- 7 items total — includes the new `e2e_remote_save_001`.

In [9]:
print("=" * 60)
print("SCENARIO 6: Remote mode — save() uploads to remote, no local file")
print("=" * 60)

REMOTE_SAVE_ITEMS = [
    {
        "id": "e2e_remote_save_001",
        "input": {"question": "Can I change my delivery address?"},
        "expected_output": {"intent": "delivery_update"},
    },
]

ds6 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)

# save() should upload to remote
ds6.save(REMOTE_SAVE_ITEMS)

# Verify the item was uploaded to remote
result6 = ds6.load()
summary6 = summarise_remote(result6)

assert summary6["count"] == 7, f"Expected 7 items, got {summary6['count']}"
assert "e2e_remote_save_001" in summary6["ids"], (
    "e2e_remote_save_001 was NOT uploaded to remote"
)

# Verify local file was NOT modified (still has 6 items from Scenario 4)
local_items_after = read_local()
local_ids_after = sorted(item["id"] for item in local_items_after)
assert len(local_items_after) == 6, (
    f"Local file should still have 6 items, got {len(local_items_after)}"
)
assert "e2e_remote_save_001" not in local_ids_after, (
    "e2e_remote_save_001 should NOT be in local file"
)

print(f"\n✓ PASSED — Remote has {summary6['count']} items (new item uploaded)")
print(f"  Local file unchanged: {len(local_items_after)} items")
print("  → Check Langfuse UI: 7 items, e2e_remote_save_001 present")

kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Uploading 1 new item(s) to remote dataset 'e2e-local-20260313-115313'.


SCENARIO 6: Remote mode — save() uploads to remote, no local file


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 7 item(s) (sync_policy='remote').



✓ PASSED — Remote has 7 items (new item uploaded)
  Local file unchanged: 6 items
  → Check Langfuse UI: 7 items, e2e_remote_save_001 present


---
## Scenario 7: Remote mode — versioned load

**Precondition:** 7 items on remote after Scenario 6.

**Steps:**
1. Record a timestamp **before** adding a new item.
2. Add a new item (`e2e_version_001`) directly via the Langfuse client to the
   existing remote dataset.
3. Record a timestamp **after** the addition.
4. Load with `version` set to the **before** timestamp — should NOT include the new item.
5. Load with `version` set to the **after** timestamp — should include the new item.
6. Load without `version` — should return latest (includes the new item).

**Expected behaviour:**
- Versioned load at the "before" timestamp returns 7 items (the new item didn't exist yet).
- Versioned load at the "after" timestamp returns 8 items.
- Unversioned load returns 8 items (latest state).

**Note:** This scenario requires `langfuse>=3.14.0`. If the SDK version is older,
the versioned `get_dataset` call will fail.

**Verify in Langfuse UI:**
- 8 items total after this scenario (the new `e2e_version_001` is visible).

In [10]:
import time
from datetime import timezone

print("=" * 60)
print("SCENARIO 7: Remote mode — versioned load")
print("=" * 60)

# Step 1: record timestamp BEFORE adding the new item
ts_before = datetime.now(tz=timezone.utc)
print(f"  Timestamp before: {ts_before.isoformat()}")

# Small delay to ensure timestamp separation
time.sleep(2)

# Step 2: add a new item directly via the Langfuse client
from langfuse import Langfuse

lf_client = Langfuse(**CREDENTIALS)
lf_client.create_dataset_item(
    dataset_name=DATASET_NAME,
    id="e2e_version_001",
    input={"question": "What are your opening hours?"},
    expected_output={"intent": "business_hours"},
)
lf_client.flush()
print("  Added e2e_version_001 to remote dataset")

# Small delay to let Langfuse process the item
time.sleep(2)

# Step 3: record timestamp AFTER the addition
ts_after = datetime.now(tz=timezone.utc)
print(f"  Timestamp after:  {ts_after.isoformat()}")

# Step 4: versioned load at "before" timestamp — should have 6 items
ds7_before = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
    version=ts_before.isoformat(),
)
result7_before = ds7_before.load()
summary7_before = summarise_remote(result7_before)

print(f"\n  Versioned load (before): {summary7_before['count']} items")
assert summary7_before["count"] == 7, (
    f"Expected 7 items at 'before' timestamp, got {summary7_before['count']}"
)
assert "e2e_version_001" not in summary7_before["ids"], (
    "e2e_version_001 should NOT be in the 'before' snapshot"
)
print("  ✓ 'before' snapshot has 7 items (new item excluded)")

# Step 5: versioned load at "after" timestamp — should have 8 items
ds7_after = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
    version=ts_after.isoformat(),
)
result7_after = ds7_after.load()
summary7_after = summarise_remote(result7_after)

print(f"  Versioned load (after):  {summary7_after['count']} items")
assert summary7_after["count"] == 8, (
    f"Expected 8 items at 'after' timestamp, got {summary7_after['count']}"
)
assert "e2e_version_001" in summary7_after["ids"], (
    "e2e_version_001 should be in the 'after' snapshot"
)
print("  ✓ 'after' snapshot has 8 items (new item included)")

# Step 6: unversioned load — should return latest (8 items)
ds7_latest = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)
result7_latest = ds7_latest.load()
summary7_latest = summarise_remote(result7_latest)

print(f"  Unversioned load:        {summary7_latest['count']} items")
assert summary7_latest["count"] == 8, (
    f"Expected 8 items (latest), got {summary7_latest['count']}"
)

print(f"\n✓ PASSED — Versioned load works correctly")
print(f"  before={ts_before.isoformat()} → 7 items")
print(f"  after ={ts_after.isoformat()} → 8 items")
print(f"  latest → 8 items")
print("  → Check Langfuse UI: 8 items total, e2e_version_001 present")

SCENARIO 7: Remote mode — versioned load
  Timestamp before: 2026-03-13T12:00:58.183723+00:00
  Added e2e_version_001 to remote dataset


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loading versioned snapshot of 'e2e-local-20260313-115313' at 2026-03-13T12:00:58.183723+00:00.
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 7 item(s) (sync_policy='remote').


  Timestamp after:  2026-03-13T12:01:02.356312+00:00

  Versioned load (before): 7 items
  ✓ 'before' snapshot has 7 items (new item excluded)


kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loading versioned snapshot of 'e2e-local-20260313-115313' at 2026-03-13T12:01:02.356312+00:00.
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 8 item(s) (sync_policy='remote').
kedro_agentic_workflows.datasets.langfuse_evaluation_dataset — INFO — Loaded dataset 'e2e-local-20260313-115313' with 8 item(s) (sync_policy='remote').


  Versioned load (after):  8 items
  ✓ 'after' snapshot has 8 items (new item included)
  Unversioned load:        8 items

✓ PASSED — Versioned load works correctly
  before=2026-03-13T12:00:58.183723+00:00 → 7 items
  after =2026-03-13T12:01:02.356312+00:00 → 8 items
  latest → 8 items
  → Check Langfuse UI: 8 items total, e2e_version_001 present


---
## Remote mode summary

If Scenarios 5–7 all printed `✓ PASSED`, remote mode is working correctly.

**Langfuse UI checklist:**

| Check | Expected |
|-------|----------|
| Total items | 8 (6 from local mode + `e2e_remote_save_001` + `e2e_version_001`) |
| `e2e_remote_save_001` present | `save()` in remote mode uploaded it (Scenario 6) |
| `e2e_version_001` present | Added in Scenario 7 |
| Local file unchanged | Still 6 items (no remote-mode writes to local) |
| Versioned snapshots | Before-timestamp excludes `e2e_version_001` |